# 2. EDA and synthetic-data audit

Inspect distributions, missingness, periodic patterns and dependencies before adding model complexity. Only the first 85% is loaded.

In [ ]:
from pathlib import Path
import json
import sys
import yaml

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))
EXPERIMENT = yaml.safe_load(
    (ROOT / "configs/synthetic_experiment.yml").read_text()
)
DATASET = ROOT / EXPERIMENT["dataset"]
RUN = ROOT / EXPERIMENT["output"]


In [ ]:
from telco_anomaly.synthetic_pipeline import load_observations
import matplotlib.pyplot as plt

data, manifest = load_observations(DATASET)
display(data.describe())
display(data.isna().mean().sort_values(ascending=False).rename("missing_fraction"))
entity = data.ont_id.iloc[0]
trace = data.loc[data.ont_id == entity].set_index("timestamp_utc")
trace[["rx_power_dbm", "olt_rx_power_dbm", "temperature_c"]].plot(
    subplots=True, figsize=(12, 7), title=entity
)
plt.tight_layout()

In [ ]:
daily = data.assign(hour=data.timestamp_utc.dt.hour).groupby("hour")
daily[["temperature_c", "throughput_mbps"]].mean().plot(
    subplots=True, figsize=(10, 5), title="UTC hourly profiles"
)
plt.tight_layout()
display(data[["temperature_c", "bias_current_ma", "tx_power_dbm",
              "rx_power_dbm", "olt_rx_power_dbm"]].corr())

In [ ]:
# Reindex to the actual grid: do not mistake dropped polls for adjacent samples.
minutes = manifest["config"]["sample_minutes"]
grid = trace.temperature_c.asfreq(f"{minutes}min")
for hours in (1, 24, 168):
    lag = round(hours * 60 / minutes)
    print(f"Temperature correlation at {hours} hours: {grid.autocorr(lag):.3f}")

Daily/weekly profiles and autocorrelation are diagnostics, not proof of stationarity. Ninety days cannot validate annual seasonality. Temperature and load have assumed periodic components; optical power is not forced to inherit a daily cycle. Seasonal conditioning should be added only if residual diagnostics justify it. Fault truth remains outside this model input table.